# Ablation Study: Baseline Evaluation
Đánh giá mBART50 gốc (chưa fine-tune) để so sánh với fine-tuned model

> **Kaggle Setup:** Bật GPU trong Settings → Accelerator → GPU T4 x2 (hoặc P100)

In [ ]:

!pip install -q evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.1 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import torch
import evaluate
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset


os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

metric = evaluate.load("sacrebleu")
print("Setup hoàn tất!")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


Setup hoàn tất!


In [3]:
# 1. Load mBART50 gốc (Baseline - chưa fine-tune)

print("Loading mBART50 baseline...")
baseline_model_name = "facebook/mbart-large-50-many-to-many-mmt"

baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_model_name)


dtype = torch.float16 if device == "cuda" else torch.float32
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    baseline_model_name,
    torch_dtype=dtype,
).to(device)
baseline_model.eval()

print("Done!")

Loading mBART50 baseline...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Done!


In [4]:
# 2. Cấu hình 3 cặp ngôn ngữ
lang_configs = [
    ("en-vi", "en_XX", "vi_VN"),
    ("en-fr", "en_XX", "fr_XX"),
    ("de-en", "de_DE", "en_XX"),
]

# Số mẫu đánh giá 
NUM_SAMPLES = 500

val_datasets = {}
for pair, src, tgt in lang_configs:
    ds = load_dataset("Helsinki-NLP/opus-100", pair, split="validation", trust_remote_code=True)
    val_datasets[pair] = ds.select(range(min(NUM_SAMPLES, len(ds))))
    print(f"  {pair}: {len(val_datasets[pair])} samples loaded")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Helsinki-NLP/opus-100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

en-vi/test-00000-of-00001.parquet:   0%|          | 0.00/137k [00:00<?, ?B/s]

en-vi/train-00000-of-00001.parquet:   0%|          | 0.00/59.0M [00:00<?, ?B/s]

en-vi/validation-00000-of-00001.parquet:   0%|          | 0.00/138k [00:00<?, ?B/s]

...

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/254k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

  de-en: 500 samples loaded


In [18]:
# 3. Hàm dịch với mBART50 gốc
def translate_baseline(texts, src_lang, tgt_lang, model, tokenizer, batch_size=8, max_len=128):
    """
    batch_size=8 an toàn hơn cho VRAM 16GB của Kaggle GPU.
    Tăng lên 16 nếu dùng GPU T4 x2 hoặc P100.
    """
    results = []
    forced_bos_id = tokenizer.lang_code_to_id[tgt_lang]
    tokenizer.src_lang = src_lang

    for i in tqdm(range(0, len(texts), batch_size), desc=f"{src_lang}→{tgt_lang}"):
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            max_length=max_len,
            truncation=True,
            padding=True,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Dùng autocast để tránh lỗi dtype khi dùng float16
        with torch.no_grad():
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                generated = model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_id,
                    max_length=max_len,
                    num_beams=4,
                )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        results.extend(decoded)

    return results

In [19]:
# 4. Đánh giá Baseline
print("=" * 60)
print("BASELINE EVALUATION (mBART50 gốc, chưa fine-tune)")
print("=" * 60)

# Mapping cặp ngôn ngữ → key trong dataset
pair_keys = {
    "en-vi": ("en", "vi"),
    "en-fr": ("en", "fr"),
    "de-en": ("de", "en"),
}

baseline_results = {}
for pair, src, tgt in lang_configs:
    ds = val_datasets[pair]
    src_key, tgt_key = pair_keys[pair]

    texts = ds["translation"]
    src_texts = [t[src_key] for t in texts]
    tgt_texts = [t[tgt_key] for t in texts]

    preds = translate_baseline(src_texts, src, tgt, baseline_model, baseline_tokenizer)
    refs = [[r] for r in tgt_texts]

    result = metric.compute(predictions=preds, references=refs)
    baseline_results[pair] = result["score"]
    print(f"  {pair}: BLEU = {result['score']:.2f}")

baseline_avg = np.mean(list(baseline_results.values()))
print(f"\n  Baseline BLEU trung bình: {baseline_avg:.2f}")

BASELINE EVALUATION (mBART50 gốc, chưa fine-tune)


en_XX→vi_VN: 100%|██████████| 63/63 [00:36<00:00,  1.72it/s]


  en-vi: BLEU = 18.07


en_XX→fr_XX: 100%|██████████| 63/63 [01:40<00:00,  1.59s/it]


  en-fr: BLEU = 33.39


de_DE→en_XX: 100%|██████████| 63/63 [01:09<00:00,  1.10s/it]

  de-en: BLEU = 35.46

  Baseline BLEU trung bình: 28.97


In [20]:
# 5. So sánh Ablation: Baseline vs Fine-tuned
# Các giá trị dưới đây lấy từ kết quả evaluate_model.py trên tập test (2000 mẫu/pair)
# với beam search (num_beams=4). Để tái lập: python -m src.evaluate_model --split test
fine_tuned_results = {
    "en-vi": 20.47,
    "en-fr": 30.07,
    "de-en": 32.53,
}
fine_tuned_avg = np.mean(list(fine_tuned_results.values()))

print("\n" + "=" * 60)
print("ABLATION STUDY: Baseline vs Fine-tuned")
print("=" * 60)
print(f"\n{'Cặp ngôn ngữ':<15} {'Baseline':>10} {'Fine-tuned':>12} {'Cải thiện':>12}")
print("-" * 52)

for pair, src, tgt in lang_configs:
    b = baseline_results[pair]
    f = fine_tuned_results[pair]
    delta = f - b
    pct = (delta / b) * 100 if b > 0 else 0
    print(f"  {pair:<13} {b:>10.2f} {f:>12.2f} {delta:>+8.2f} ({pct:>+.1f}%)")

print("-" * 52)
delta_avg = fine_tuned_avg - baseline_avg
pct_avg = (delta_avg / baseline_avg) * 100 if baseline_avg > 0 else 0
print(f"  {'Trung bình':<13} {baseline_avg:>10.2f} {fine_tuned_avg:>12.2f} {delta_avg:>+8.2f} ({pct_avg:>+.1f}%)")

print("\n" + "=" * 60)
print("KẾT LUẬN")
print("=" * 60)
print(f"  Fine-tuning cải thiện BLEU trung bình +{delta_avg:.2f} điểm ({pct_avg:+.1f}%)")
best_pair = max(fine_tuned_results, key=lambda p: fine_tuned_results[p] - baseline_results[p])
print(f"  Cặp '{best_pair}' hưởng lợi nhiều nhất từ fine-tuning")


ABLATION STUDY: Baseline vs Fine-tuned

Cặp ngôn ngữ      Baseline   Fine-tuned    Cải thiện
----------------------------------------------------
  en-vi              18.07        20.47    +2.40 (+13.3%)
  en-fr              33.39        30.07    -3.32 (-9.9%)
  de-en              35.46        32.53    -2.93 (-8.3%)
----------------------------------------------------
  Trung bình         28.97        27.69    -1.28 (-4.4%)

KẾT LUẬN
  Fine-tuning cải thiện BLEU trung bình +-1.28 điểm (-4.4%)
  Cặp 'en-vi' hưởng lợi nhiều nhất từ fine-tuning
